# FLAG 2027 - Notebook 38 RUN-ALL: the same-gender problem on its own (research round 8)

Our random-gender cells (16.5 / 16.2) are better than anything published; the whole gap to the teams above us is the two same-gender cells (29.5 / 26.4). Three literature agents on 5 Nov looked only at that (summary in `research/round8_2026-11-05.md`). Two findings shaped this notebook: attribute supervision never helped the gender protocol in any paper (DIMNet: gender +0.1, nationality worse), and the methods that shrank the gap all changed the *geometry of training*: which identities meet in a batch, and how hard impostors and interior points are pulled.

| Name | What | Precedent |
|---|---|---|
| `S_conf8` | batches of 2 seed identities + the 7 nearest identities of each, nearest = frozen-feature centroid cosine (face and voice) | Hard Prototype Mining (IDLab 2020), CHNS (Interspeech 2025): âˆ’15 to âˆ’24% EER on hard trial lists, loss unchanged |
| `S_conf4` | 4 seeds x 4 confusable identities | same |
| `S_half` | 1 seed + 7 neighbours, 8 random (CHNS hard ratio 0.5) | same |
| `L_convex` | a random point between the face and voice embedding of each row is a third sample of that identity in CE, OPL and L_out | Kim & Kang 2026: gap random-to-gender 8.2 vs FOP 14.7 on the same split |
| `L_ms` / `L_ms_add` | multi-similarity loss on the cross-modal cosine matrix, replacing or added to L_out | Fuse-after-Align 2024, Xiaofei FAME 2024 |
| `L_panchor` | prototype-anchored hinge: own prototype must beat the hardest other-identity row | CMC 2025 simplex-center paper (+3.5 on the gender protocol) |
| `W_drop10` | the 10% of training identities with the highest running CE loss are excluded from sampling after epoch 8 | Wen et al. CVPR 2021 identity reweighting |

Reference `R0_ref` = system E's recipe (copied from NB36). Stage A = usual folds, seeds 1-3; top 3 re-checked on NEW folds with NEW seeds (stage B); only stage B counts. Stage C combines whatever stage B adopts.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 1-1.5 h. Resumable. Paste the **SUMMARY**.


In [ ]:
# Cell 1 - Drive, GPU (hard check)
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
print('[NB38 v1 5-Nov] Device:', torch.cuda.get_device_name(0), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - features, labels, CV folds (identical to NB24), harness with the new recipe options
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# NB38 - research round 8: the same-gender problem on its own (three literature agents, 5 Nov; summary in research/round8_2026-11-05.md)
# Every candidate changes HOW the head is trained, not what it sees: batch composition, extra samples, or the loss geometry around hard impostors.
#   S_conf8   : confusable-identity batches (HPM 2020 / CHNS 2025): 2 seed identities + the 7 nearest identities of each (frozen-feature centroids)
#   S_conf4   : 4 seeds x 4 confusable identities
#   S_half    : 1 seed + 7 neighbours, other 8 identities random (CHNS "hard ratio" 0.5)
#   L_convex  : cross-modal convex interpolation (Kim & Kang 2026): c = a*face + (1-a)*voice, a ~ U(0,1), added as a third sample of the identity to CE, OPL and L_out
#   L_ms      : multi-similarity loss (Wang 2019; Fuse-after-Align 2024) on the cross-modal cosine matrix INSTEAD of L_out
#   L_ms_add  : multi-similarity loss ADDED to L_out (w 0.25)
#   L_panchor : prototype-anchored hinge (CMC 2025): own prototype vs the hardest other-identity sample in the batch, both modalities, w 0.2
#   W_drop10  : identity reweighting (Wen CVPR 2021): the 10% of training identities with the highest running CE loss are excluded from sampling after epoch 8
# Reference R0_ref = system E's recipe (CE + 2*OPL + 0.5*L_out, tau 0.07), copied from NB36 M0_ref. Stage A usual folds seeds 1-3; stage B new folds + seeds 4-6 on the top 3;
# stage C = combinations of whatever stage B adopts. Training-side only; per-trial scoring; no test statistics.
SUMMARY = []
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def ms_loss(ft, vt, lab, a=2.0, b=50.0, lam=0.5, eps=0.1):
    """multi-similarity loss on the face x voice cosine matrix, both directions; pair mining relative to the hardest positive / negative (Wang et al. 2019)"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t(); same = lab[:, None] == lab[None, :]
    def one(S_, same_):
        min_pos = S_.masked_fill(~same_, 1e4).min(1, keepdim=True)[0]; max_neg = S_.masked_fill(same_, -1e4).max(1, keepdim=True)[0]
        pm = (same_ & (S_ < max_neg + eps)).float(); nm = ((~same_) & (S_ > min_pos - eps)).float()
        lp = torch.log1p((torch.exp(-a * (S_ - lam)) * pm).sum(1)) / a
        ln = torch.log1p((torch.exp(b * (S_ - lam)) * nm).sum(1)) / b
        return (lp + ln).mean()
    return 0.5 * (one(S, same) + one(S.t(), same.t()))
def panchor_loss(model, ft, vt, lab, m=0.3):
    """prototype-anchored hinge: for each row, cosine distance to its own prototype must beat the closest OTHER-identity row to that prototype by m"""
    out = 0.0; same = lab[:, None] == lab[None, :]
    for z in (ft, vt):
        C = Fn.normalize(model.cls(z), dim=1) @ model.P.t()                    # B x n_cls cosines in prototype space
        d_pos = 1 - C.gather(1, lab[:, None]).squeeze(1)                        # distance to own prototype
        Dp = 1 - C[:, lab]                                                       # Dp[j, i] = distance of row j to the prototype of row i
        d_neg = Dp.masked_fill(same, 1e4).min(0)[0]                              # hardest other-identity row per prototype
        out = out + Fn.relu(m + d_pos - d_neg).mean()
    return out
def conf_pool(XF, XV, tr, ytr_np, n_cls, k=32):
    """per-identity centroids of the FROZEN, domain-normalised features; cosine confusability = mean of face and voice; k nearest identities each"""
    with torch.no_grad():
        yt = torch.tensor(ytr_np, device=dev); cnt = torch.bincount(yt, minlength=n_cls).float()[:, None]
        Fc = torch.zeros(n_cls, XF.shape[1], device=dev).index_add_(0, yt, XF[tr]) / cnt
        Vc = torch.zeros(n_cls, XV.shape[1], device=dev).index_add_(0, yt, XV[tr]) / cnt
        S = 0.5 * (Fn.normalize(Fc, dim=1) @ Fn.normalize(Fc, dim=1).t()) + 0.5 * (Fn.normalize(Vc, dim=1) @ Fn.normalize(Vc, dim=1).t())
        S.fill_diagonal_(-1e9)
        return S.topk(k, dim=1).indices.cpu().numpy()
def sample_ids(rng, avail, nbr, n_seed, I, n_rand):
    """n_seed seeds, each with I-1 of its confusable neighbours, plus n_rand random identities; all distinct, all from avail"""
    chosen = []; ok = avail.copy()
    for _ in range(n_seed):
        s = rng.choice(np.where(ok)[0]); chosen.append(s); ok[s] = False
        pool = [n for n in nbr[s] if ok[n]]; take = list(rng.choice(pool, min(I - 1, len(pool)), replace=False)) if pool else []
        if len(take) < I - 1:
            take += list(rng.choice(np.where(ok & ~np.isin(np.arange(len(ok)), take))[0], I - 1 - len(take), replace=False))
        for n in take:
            chosen.append(n); ok[n] = False
    if n_rand > 0:
        for n in rng.choice(np.where(ok)[0], n_rand, replace=False):
            chosen.append(n)
    return np.array(chosen)
def train_head8(F_raw, V_raw, all_tr, vsets, seed, cfg):
    w_al, tau = 0.5, 0.07
    conf, convex, ms, w_pa, drop = cfg.get('conf'), cfg.get('convex', False), cfg.get('ms'), cfg.get('panchor', 0.0), cfg.get('drop', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}; n_cls = len(cl)
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(n_cls)]
    tr = torch.tensor(all_tr, device=dev); model = FOP(n_cls, XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    nbr = conf_pool(XF, XV, tr, ytr_np, n_cls) if conf else None
    ema = np.full(n_cls, np.nan); avail = np.ones(n_cls, bool)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        if drop > 0 and ep > 8:
            seen = ~np.isnan(ema); avail[:] = True
            if seen.sum() > 20:
                cut = np.quantile(ema[seen], 1 - drop); avail[seen & (np.nan_to_num(ema, nan=-1.0) >= cut)] = False
        for _ in range(STEPS):
            if conf:
                chosen = sample_ids(rng, avail, nbr, conf[0], conf[1], conf[2])
            else:
                chosen = rng.choice(np.where(avail)[0], P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]); j = torch.tensor(j_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[j]] - mv) / sv; lab = ytr[j]
            fused, logits, ft, vt = model(xf, xv)
            ce_rows = Fn.cross_entropy(logits, lab, reduction='none')
            if drop > 0:
                with torch.no_grad():
                    lc, li = lab.cpu().numpy(), ce_rows.detach().cpu().numpy()
                    for c in np.unique(lc):
                        v_ = li[lc == c].mean(); ema[c] = v_ if np.isnan(ema[c]) else 0.9 * ema[c] + 0.1 * v_
            if convex:
                a_ = torch.rand(len(lab), 1, device=dev); c_ = a_ * ft + (1 - a_) * vt; lab2 = torch.cat([lab, lab])
                ce = 0.5 * (ce_rows.mean() + Fn.cross_entropy(model.cls(c_) @ model.P.t(), lab))
                loss = ce + ALPHA * opl_loss(torch.cat([fused, c_]), lab2) + w_al * lout_loss(torch.cat([ft, c_]), torch.cat([vt, c_]), lab2, tau)
            else:
                loss = ce_rows.mean() + ALPHA * opl_loss(fused, lab)
                if ms == 'replace':
                    loss = loss + w_al * ms_loss(ft, vt, lab)
                elif ms == 'add':
                    loss = loss + w_al * lout_loss(ft, vt, lab, tau) + 0.25 * ms_loss(ft, vt, lab)
                else:
                    loss = loss + w_al * lout_loss(ft, vt, lab, tau)
            if w_pa > 0:
                loss = loss + w_pa * panchor_loss(model, ft, vt, lab)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
CFGS = {'S_conf8': dict(conf=(2, 8, 0)), 'S_conf4': dict(conf=(4, 4, 0)), 'S_half': dict(conf=(1, 8, 8)),
        'L_convex': dict(convex=True), 'L_ms': dict(ms='replace'), 'L_ms_add': dict(ms='add'), 'L_panchor': dict(panchor=0.2), 'W_drop10': dict(drop=0.1)}
# sanity on the confusability pool: how often is the nearest identity of the same gender? (challenge identities only, fold 0 training rows)
_tr0 = np.concatenate([FOLD_DATA[0][0], EXT]); _F, _V, _ = dnorm(F_RAW, V_RAW, _tr0); _cl = np.unique(Y_ALL[_tr0]); _rm = {c: i for i, c in enumerate(_cl)}
_nbr = conf_pool(torch.tensor(_F, device=dev), torch.tensor(_V, device=dev), torch.tensor(_tr0, device=dev), np.array([_rm[c] for c in Y_ALL[_tr0]]), len(_cl))
_ch = [i for i, c in enumerate(_cl) if c < 1000]; _sg = np.mean([id_gender[_cl[i]] == id_gender[_cl[_nbr[i, 0]]] for i in _ch if _cl[_nbr[i, 0]] < 1000])
_rand = np.mean([id_gender[_cl[i]] == id_gender[_cl[j]] for i in _ch for j in np.random.default_rng(0).choice(_ch, 3)])
note('confusability pool: the nearest identity has the same gender %.0f%% of the time (random pairs: %.0f%%)' % (100 * _sg, 100 * _rand))
del _F, _V, _nbr
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV38 = os.path.join(RESDIR, 'nb38_cv.csv')
if not os.path.exists(CSV38):
    r = pd.read_csv(os.path.join(RESDIR, 'nb36_cv.csv')); r = r[r.combo == 'M0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    r['combo'] = 'R0_ref'; r.to_csv(CSV38, index=False)
    print('R0_ref copied from NB36 M0_ref (%d runs: stage A %d, stage B %d)' % (len(r), (r.stage == 'A').sum(), (r.stage == 'B').sum()))
done38 = pd.read_csv(CSV38); t0 = time.time()
def run38(stage, names, seeds, fold_data):
    global done38
    for name in names:
        for seed in seeds:
            for k in range(K):
                if ((done38.stage == stage) & (done38.combo == name) & (done38.seed == seed) & (done38.fold == k)).any():
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_head8(F_RAW, V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed, CFGS[name])
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV38, mode='a', header=False, index=False); done38 = pd.concat([done38, row])
            print('%s %-10s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
def tab38(stage, names=None):
    R = pd.read_csv(CSV38); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    if names is not None:
        R = R[R.combo.isin(list(names) + ['R0_ref'])]
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def adopt(TB, c):
    refB = TB.loc['R0_ref']; dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    return (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)), dB, ds, dg
run38('A', list(CFGS), [1, 2, 3], FOLD_DATA)
TA = tab38('A'); refA = TA.loc['R0_ref']; TOP = [c for c in TA.index if c != 'R0_ref'][:3]
run38('B', TOP, [4, 5, 6], FOLD_B)
TB = tab38('B')
note('NB38 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe from NB36):'); note(TA.to_string())
note('NB38 STAGE B (NEW folds + NEW seeds 4-6, top 3 of stage A):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
WIN = []
for c in TOP:
    ok, dB, ds, dg = adopt(TB, c)
    note('%-10s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, json.dumps(CFGS[c]), TA.loc[c, 'overall'] - refA['overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    if ok:
        WIN.append(c)
# stage C: combine the adopted ingredients (sampling + loss changes compose; two loss replacements do not)
if len(WIN) >= 2:
    combo = {}
    for c in WIN:
        combo.update(CFGS[c])
    name = 'C_' + '+'.join(WIN); CFGS[name] = combo
    run38('A', [name], [1, 2, 3], FOLD_DATA); run38('B', [name], [4, 5, 6], FOLD_B)
    TAc, TBc = tab38('A', [name]), tab38('B', [name]); ok, dB, ds, dg = adopt(TBc, name)
    note('STAGE C combination %s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (name, json.dumps(combo), TAc.loc[name, 'overall'] - TAc.loc['R0_ref', 'overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    for c in WIN:
        note('  vs %s alone on stage B: %+.2f' % (c, TBc.loc[name, 'overall'] - TB.loc[c, 'overall']))
elif len(WIN) == 1:
    note('one adopted ingredient (%s); no combination to test' % WIN[0])
else:
    note('nothing adopted; no combination to test')
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
